# Advanced Neural Operators

## Basic Fourier Neural Operator (FNO)
This guide explains how to implement neural operator architectures for solving PDEs using PhysicsNeMo. Neural operators learn mappings between function spaces and can generalize to different resolutions and parameters.

We'll work with the **2D Reaction-Diffusion equation** on the unit square $[0,1]^2$:

$$u(x, y) - \Delta u(x, y) = f(x, y)$$

with periodic boundary conditions. We construct exact solutions using Fourier series, making this an ideal testbed for operator learning.

## Mathematical Formulation

### Problem Setup
- **PDE**: Reaction-Diffusion equation $u - \Delta u = f$
- **Domain**: Unit square $[0,1]^2$ with periodic BCs
- **Exact Solution**: Solve for $u$ from random Fourier source $f$
- **Data**: Generate synthetic $(f, u)$ pairs where $u = (1 - \Delta)^{-1} f$
- **Goal**: Learn the operator $\mathcal{G}: f \mapsto u$
- **Note**: $u$ and $f$ naturally have similar scales — no normalization needed!

---

## Step 0: Data Generation

Before training any model, we need to generate the dataset using Fourier series.

### Generate Dataset

Run the data generation script:

```bash
python generate_data.py
```

This creates:
- `datasets/Reaction_Diffusion/train.hdf5` (8000 samples)
- `datasets/Reaction_Diffusion/val.hdf5` (1000 samples)
- `datasets/Reaction_Diffusion/test.hdf5` (1000 samples)

### Data Generation Method

The script generates exact solutions using Fourier series:

Step 1: Construct f as random Fourier series
$$f(x,y) = \sum_{k,l} \text{coeff}[k,l] * \sin(2\pi kx) * \sin(2\pi ly)$$

Step 2: Solve for u analytically using $u - \Delta u = f$
$$u(x,y) = \sum_{k,l} \frac{\text{coeff}[k,l]}{1 + (2\pi)^2(k^2 + l^2)} * \sin(2\pi kx) * \sin(2\pi ly)$$


In [ ]:
!python generate_data.py

## Level 1 Fourier Neural Operator

FNO operates in the Fourier domain to capture global correlations efficiently.

### Theory

**FNO Layer**:
1. Transform to Fourier space: $\mathcal{F}(v)$
2. Multiply by learned weights in frequency domain
3. Transform back: $\mathcal{F}^{-1}(W \cdot \mathcal{F}(v))$
4. Add skip connection and activation

**Key Advantage**: $O(N \log N)$ complexity via FFT

### Implementation with PhysicsNeMo

The `fno_physicsnemo_l1.py` script uses PhysicsNeMo's built-in FNO architecture:

```python
import physicsnemo.sym
from physicsnemo.sym.hydra import instantiate_arch, PhysicsNeMoConfig
from physicsnemo.sym.key import Key
from physicsnemo.sym.domain import Domain
from physicsnemo.sym.domain.constraint import SupervisedGridConstraint
from physicsnemo.sym.domain.validator import GridValidator
from physicsnemo.sym.dataset import DictGridDataset
from physicsnemo.sym.solver import Solver

@physicsnemo.sym.main(config_path="conf", config_name="config_FNO")
def run(cfg: PhysicsNeMoConfig) -> None:
    train_file = to_absolute_path("datasets/Reaction_Diffusion/train.hdf5")
    test_file = to_absolute_path("datasets/Reaction_Diffusion/test.hdf5")
    
    # Define input/output keys (no normalization needed!)
    input_keys = [Key("f")]
    output_keys = [Key("u")]
    
    # Load data
    invar_train, outvar_train = load_dataset(
        FIXME, 
        [k.name for k in input_keys],
        [k.name for k in output_keys],
    )
    invar_test, outvar_test = load_dataset(
        FIXME,
        [k.name for k in input_keys],
        [k.name for k in output_keys],
    )
    
    # Create datasets
    # Hint: use DictGridDataset
    FIXME
    
    # Create FNO model
    # Hint: use instantiate_arch
    FIXME
    
    # Setup domain and constraints
    domain = Domain()
    supervised = SupervisedGridConstraint(
        nodes=nodes,
        dataset=train_dataset,
        batch_size=cfg.batch_size.grid,
    )
    domain.add_constraint(supervised, "supervised")
    
    # Add validator
    val = GridValidator(
        nodes,
        dataset=test_dataset,
        batch_size=cfg.batch_size.validation,
        plotter=GridValidatorPlotter(n_examples=5),
    )
    domain.add_validator(val, "test")
    
    # Train
    slv = Solver(cfg, domain)
    slv.solve()
```


1. Fill in all missing parts
2. Place the configuration file in the conf directory
3. Run the script:

In [ ]:
!python fno_physicsnemo_l1.py

## Level 2 Adaptive Fourier Neural Operator

AFNO extends FNO with **adaptive frequency mixing** and **patch-based processing**.

### Theory

**Key Differences from FNO**:
- **FNO**: Fixed weight multiplication in frequency domain
- **AFNO**: Adaptive token mixing with learned transformations

**AFNO Architecture**:
1. Divide input into patches
2. Transform patches to frequency domain
3. Apply learned mixing (like attention in transformers)
4. Reconstruct output

### Implementation with PhysicsNeMo

The `fno_physicsnemo_l2.py` script uses PhysicsNeMo's AFNO architecture:

```python
@physicsnemo.sym.main(config_path="conf", config_name="config_AFNO")
def run(cfg: PhysicsNeMoConfig) -> None:
    # Load data (same as Level 1)
    invar_train, outvar_train = load_dataset(
        FIXME, 
        [k.name for k in input_keys],
        [k.name for k in output_keys],
    )
    invar_test, outvar_test = load_dataset(
        FIXME,
        [k.name for k in input_keys],
        [k.name for k in output_keys],
    )
    
    # Get image shape
    img_shape = [
        next(iter(invar_train.values())).shape[-2],
        next(iter(invar_train.values())).shape[-1],
    ]
    
    # AFNO requires dimensions divisible by patch_size
    img_shape = [s - s % cfg.arch.afno.patch_size for s in img_shape]
    print(f"Cropped image shape: {img_shape}")
    
    # Crop data to match
    for d in (invar_train, outvar_train, invar_test, outvar_test):
        for k in d:
            d[k] = d[k][:, :, :img_shape[0], :img_shape[1]]
    
    # Create datasets (same as Level 1)
    # Hint: use DictGridDataset
    FIXME
    
    # Create AFNO model
    # Hint: use instantiate_arch
    FIXME
    
    # Training setup (same as FNO)
    domain = Domain()
    supervised = SupervisedGridConstraint(nodes=nodes, dataset=train_dataset, ...)
    domain.add_constraint(supervised, "supervised")
    
    # Train
    slv = Solver(cfg, domain)
    slv.solve()
```


1. Fill in all missing parts
2. Place the configuration file in the conf directory
3. Run the script:

In [ ]:
!python fno_physicsnemo_l2.py

## Level 3 Physics-Informed Neural Operator

PINO combines **data-driven learning** with **physics constraints** for better generalization.

### Theory

**Combined Loss Function**:
$$\mathcal{L} = \mathcal{L}_{\text{data}} + \lambda \mathcal{L}_{\text{physics}}$$

where:
- $\mathcal{L}_{\text{data}} = ||u_{\text{pred}} - u_{\text{true}}||^2$ (supervised loss)
- $\mathcal{L}_{\text{physics}} = ||u_{\text{pred}} - \Delta u_{\text{pred}} - f||^2$ (PDE residual)
- $\lambda$: weight balancing data and physics

### Computing the Physics Loss

For the Reaction-Diffusion equation, we compute the Laplacian using spectral methods:

```python
class ReactionDiffusionPDE(torch.nn.Module):
    """Compute PDE residual: u - Δu - f = 0"""
    
    def forward(self, input_var: Dict[str, torch.Tensor]):
        u = input_var["u"]
        f = input_var["f"]
        
        FIXME # Fill in the computation of pde_residual
        
        return {"pde_residual": pde_residual}
```

### Implementation with PhysicsNeMo

The `fno_physicsnemo_l3.py` script adds physics constraints to FNO:

```python
@physicsnemo.sym.main(config_path="conf", config_name="config_PINO")
def run(cfg: PhysicsNeMoConfig) -> None:
    # Load data (same as Level 1)
    invar_train, outvar_train = load_dataset(
        FIXME, 
        [k.name for k in input_keys],
        [k.name for k in output_keys],
    )
    invar_test, outvar_test = load_dataset(
        FIXME,
        [k.name for k in input_keys],
        [k.name for k in output_keys],
    )
    
    # Add physics constraint target (should be zero)
    outvar_train["pde_residual"] = np.zeros_like(outvar_train["u"])
    
    # Create datasets (same as Level 1)
    # Hint: use DictGridDataset
    FIXME

    # Create FNO backbone for PINO
    # Hint: use instantiate_arch
    FIXME
    
    # Create custom PDE node
    pde_node = Node(
        inputs=["u", "f"],
        outputs=["pde_residual"],
        evaluate=ReactionDiffusionPDE(),
        name="Reaction-Diffusion PDE Node",
    )
    nodes = [fno.make_node("fno"), pde_node]
    
    # Setup domain with physics constraint
    domain = Domain()
    supervised = SupervisedGridConstraint(
        nodes=nodes,
        dataset=train_dataset,  # Includes both u and pde_residual targets
        batch_size=cfg.batch_size.grid,
    )
    domain.add_constraint(supervised, "supervised")
    
    # Train
    slv = Solver(cfg, domain)
    slv.solve()
```


1. Fill in all missing parts
2. Place the configuration file in the conf directory
3. Run the script:

In [ ]:
!python fno_physicsnemo_l3.py

## Summary and Comparison

### Architecture Comparison

| Feature | FNO | AFNO | PINO |
|---------|-----|------|------|
| **Frequency Mixing** | Fixed weights | Adaptive mixing | Fixed weights |
| **Physics Constraints** | ❌ | ❌ | ✅ |
| **Processing** | Global FFT | Patch-based | Global FFT |
| **Computational Cost** | Low | Medium | Medium |
| **Data Requirements** | High | Medium | Low |
| **Physical Consistency** | Not guaranteed | Not guaranteed | Enforced |

### When to Use Each Method

**FNO (Fourier Neural Operator)**
- ✅ Strong baseline for operator learning
- ✅ Abundant training data available
- ✅ Smooth solutions
- ✅ Fast inference critical

**AFNO (Adaptive Fourier Neural Operator)**
- ✅ High-resolution data
- ✅ Large-scale problems
- ✅ Complex frequency patterns
- ✅ Weather/climate modeling

**PINO (Physics-Informed Neural Operator)**
- ✅ Known governing equations
- ✅ Limited training data
- ✅ Physical consistency critical
- ✅ Trustworthy extrapolation needed
- ✅ Scientific/engineering applications



## References and Further Reading

### Original Papers

1. **FNO**: Li et al. (2020), "Fourier Neural Operator for Parametric PDEs"
   - [arXiv:2010.08895](https://arxiv.org/abs/2010.08895)

2. **AFNO**: Guibas et al. (2021), "Adaptive Fourier Neural Operators"
   - [arXiv:2111.13587](https://arxiv.org/abs/2111.13587)

3. **PINO**: Li et al. (2021), "Physics-Informed Neural Operator"
   - [arXiv:2111.03794](https://arxiv.org/abs/2111.03794)

### Resources

- **NVIDIA PhysicsNeMo**: [github.com/NVIDIA/physicsnemo-sym](https://github.com/NVIDIA/physicsnemo-sym)
- **PhysicsNeMo Darcy Example**: [Darcy Flow](https://github.com/NVIDIA/physicsnemo-sym/tree/main/examples/darcy)


--- 

Don't forget to check out additional [Open Hackathons Resources](https://www.openhackathons.org/s/technical-resources) and join our [OpenACC and Hackathons Slack Channel](https://www.openacc.org/community#slack) to share your experience and get more help from the community.

---

# Licensing

Copyright © 2026 OpenACC-Standard.org. This material is released by OpenACC-Standard.org, in collaboration with NVIDIA Corporation, under the Creative Commons Attribution 4.0 International (CC BY 4.0). These materials may include references to hardware and software developed by other entities; all applicable licensing and copyrights apply.
